# 00 — Data Collection & Processing

Master notebook for acquiring and processing all wage-related datasets.

**Data Sources:**
1. PFD Personal Attendant Base Wage Calculator (Excel) — THE primary source
2. PFD LTSS Rate Tables (Excel)
3. BLS OEWS via API — Texas direct care worker wages
4. Market comparison wages from news/public reporting

**Key Finding:** HHSC literally publishes a spreadsheet with `$10.60/hr` as the default
target wage for personal attendants, and shows how every Medicaid service rate
is built on top of that assumption.

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
import requests
import json
from pathlib import Path

RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

---
## 1. PFD Personal Attendant Base Wage Calculator

Source: https://pfd.hhs.texas.gov/rate-tables  
File: `ltss-personal-attendant-base-wage-calculator.xlsx`  
Updated: 2/7/2025  

This is the smoking gun. HHSC's own spreadsheet shows:
- Default target wage: **$10.60/hr** (cell B7)
- Every service rate has a built-in "attendant cost" component
- Column X shows the wage assumption baked into each rate
- You can change B7 to see the fiscal impact of a wage increase

In [ ]:
# Load the Fiscal Calculation sheet — this has the per-service rate breakdown
wage_calc_path = RAW / 'rates' / 'ltss-personal-attendant-base-wage-calculator.xlsx'
wb = openpyxl.load_workbook(wage_calc_path, data_only=True)

# Extract the target wage from cell B7
fiscal_sheet = wb['Fiscal by Program']
target_wage = fiscal_sheet['B7'].value
ptb_pct = fiscal_sheet['B8'].value  # payroll taxes & benefits %
print(f"HHSC Default Target Hourly Wage: ${target_wage}")
print(f"Payroll Taxes & Benefits %: {ptb_pct:.1%}")
print(f"FMAP: State share ~40.6%, Federal share ~59.4%")

In [ ]:
# Extract the full Fiscal Calculation data
ws = wb['Fiscal Calculation']

# Build DataFrame from the detailed calculation sheet
rows = []
for row in ws.iter_rows(min_row=3, max_row=ws.max_row, values_only=False):
    cells = {c.column_letter: c.value for c in row}
    if cells.get('E') and cells.get('Q'):  # has program name and rate
        rows.append({
            'bill_code': cells.get('A'),
            'program': cells.get('E'),
            'service_category': cells.get('F'),
            'service_description': cells.get('G'),
            'bill_description': cells.get('H'),
            'is_cds': cells.get('I', '') == 'Yes',
            'is_facility_based': cells.get('D', '') == 'Yes',
            'funding_authority': cells.get('K'),
            'unit_measure': cells.get('L'),
            'units': cells.get('M'),
            'payment': cells.get('O'),
            'current_rate': cells.get('Q'),
            'attendant_cost': cells.get('R'),
            'attendant_cost_pct': cells.get('S'),
            'attendant_hours_per_unit': cells.get('T'),
            'wage_required': cells.get('X'),
            'rate_after_increase': cells.get('Y'),
            'rate_effective_date': cells.get('P'),
            'sfy26_est_hours': cells.get('AM'),
            'sfy27_est_hours': cells.get('AN'),
        })

df_rates = pd.DataFrame(rows)
print(f"Total service lines: {len(df_rates)}")
print(f"\nPrograms: {df_rates['program'].unique()}")
print(f"\nSample wage assumptions by program:")
df_rates.groupby('program')['wage_required'].describe()[['count','mean','min','max']].round(2)

In [ ]:
# Focus on the programs we care about
target_programs = ['HCS', 'ICF/IID', 'TxHmL', 'CAS', 'CLASS']
df_target = df_rates[df_rates['program'].isin(target_programs)].copy()

# Show HCS Supervised Living rates (the residential homes)
print("=== HCS Supervised Living & Residential Support ===")
hcs_residential = df_target[
    (df_target['program'] == 'HCS') & 
    (df_target['service_description'].str.contains('Supervised Living|Residential', na=False))
].drop_duplicates(subset=['current_rate', 'attendant_cost'])

for _, row in hcs_residential.iterrows():
    print(f"  {row['bill_description'][:60]:<60} Rate: ${row['current_rate']:>8.2f}/day  "
          f"Att Cost: ${row['attendant_cost']:>8.2f}  Wage Req: ${row['wage_required']:>8.2f}")

In [ ]:
# Save processed rate data
df_rates.to_csv(PROCESSED / 'pfd_wage_calculator_all_services.csv', index=False)
df_target.to_csv(PROCESSED / 'pfd_wage_calculator_target_programs.csv', index=False)
print(f"Saved {len(df_rates)} total service lines and {len(df_target)} target program lines")

---
## 2. PFD LTSS Rate Tables

Source: https://pfd.hhs.texas.gov/rate-tables  
File: `2026-2027-ltss-rte-tables.xlsx`

In [ ]:
ltss_path = RAW / 'rates' / '2026-2027-ltss-rte-tables.xlsx'
wb_ltss = openpyxl.load_workbook(ltss_path, data_only=True)
print(f"LTSS Rate Table sheets: {wb_ltss.sheetnames}")

# Explore each sheet to find HCS and ICF rates
for name in wb_ltss.sheetnames:
    ws = wb_ltss[name]
    print(f"\n--- {name} (rows: {ws.max_row}) ---")
    # Show first few rows to understand structure
    for row in ws.iter_rows(min_row=1, max_row=min(5, ws.max_row), values_only=False):
        vals = [c.value for c in row if c.value is not None]
        if vals:
            print(f"  {vals[:5]}")

---
## 3. BLS OEWS Wage Data (Texas, May 2024)

Source: BLS Public API v2  
SOC Codes:
- 31-1120: Home Health & Personal Care Aides
- 31-1131: Nursing Assistants
- 39-9011: Childcare Workers (comparison)

Latest release: May 2024 data (published March 2025)

In [ ]:
# BLS OEWS Series ID format for Texas (area code 48 = Texas)
# Data type codes: 01=employment, 03=hourly mean, 04=annual mean,
#                  06=10th pctile, 07=25th pctile, 08=median,
#                  11=75th pctile annualized, 12=90th pctile annualized, 13=median annualized

series_map = {
    # Home Health & Personal Care Aides (31-1120)
    'OEUS480000000000031112001': ('31-1120', 'Home Health and Personal Care Aides', 'employment'),
    'OEUS480000000000031112003': ('31-1120', 'Home Health and Personal Care Aides', 'hourly_mean'),
    'OEUS480000000000031112004': ('31-1120', 'Home Health and Personal Care Aides', 'annual_mean'),
    'OEUS480000000000031112006': ('31-1120', 'Home Health and Personal Care Aides', 'pct10_hourly'),
    'OEUS480000000000031112007': ('31-1120', 'Home Health and Personal Care Aides', 'pct25_hourly'),
    'OEUS480000000000031112008': ('31-1120', 'Home Health and Personal Care Aides', 'median_hourly'),
    # Nursing Assistants (31-1131)
    'OEUS480000000000031113101': ('31-1131', 'Nursing Assistants', 'employment'),
    'OEUS480000000000031113103': ('31-1131', 'Nursing Assistants', 'hourly_mean'),
    'OEUS480000000000031113104': ('31-1131', 'Nursing Assistants', 'annual_mean'),
    # Childcare Workers (39-9011)
    'OEUS480000000000039901103': ('39-9011', 'Childcare Workers', 'hourly_mean'),
    'OEUS480000000000039901104': ('39-9011', 'Childcare Workers', 'annual_mean'),
}

# Pull from BLS API
response = requests.post(
    'https://api.bls.gov/publicAPI/v2/timeseries/data/',
    json={
        'seriesid': list(series_map.keys()),
        'startyear': '2024',
        'endyear': '2024',
    },
    timeout=30,
)
bls_data = response.json()

# Parse results
bls_rows = []
for series in bls_data['Results']['series']:
    sid = series['seriesID']
    soc, title, measure = series_map.get(sid, ('?', '?', '?'))
    for rec in series['data']:
        bls_rows.append({
            'year': int(rec['year']),
            'soc_code': soc,
            'occupation': title,
            'measure': measure,
            'value': float(rec['value']),
        })

df_bls = pd.DataFrame(bls_rows)
print("=== BLS OEWS Texas Wage Data (May 2024) ===")
print(df_bls.pivot_table(index=['soc_code', 'occupation'], columns='measure', values='value').to_string())

In [ ]:
# Save BLS data
df_bls.to_csv(PROCESSED / 'bls_oews_texas_2024.csv', index=False)

# Key comparison
hha_mean = df_bls[(df_bls['soc_code'] == '31-1120') & (df_bls['measure'] == 'hourly_mean')]['value'].iloc[0]
hha_median = df_bls[(df_bls['soc_code'] == '31-1120') & (df_bls['measure'] == 'median_hourly')]['value'].iloc[0]
hhsc_wage = 10.60
hhsc_new = 13.00

print(f"\n=== The Gap ===")
print(f"HHSC default target wage:        ${hhsc_wage:.2f}/hr")
print(f"HHSC new target wage:            ${hhsc_new:.2f}/hr")
print(f"BLS TX Home Health Aide mean:    ${hha_mean:.2f}/hr")
print(f"BLS TX Home Health Aide median:  ${hha_median:.2f}/hr")
print(f"BLS TX Home Health Aide 10th pct: ${df_bls[(df_bls['soc_code'] == '31-1120') & (df_bls['measure'] == 'pct10_hourly')]['value'].iloc[0]:.2f}/hr")
print(f"\nEven the BLS MARKET mean (${hha_mean:.2f}) is below the $13 target.")
print(f"The 10th percentile (${df_bls[(df_bls['soc_code'] == '31-1120') & (df_bls['measure'] == 'pct10_hourly')]['value'].iloc[0]:.2f}) is below the OLD $10.60 assumption.")

---
## 4. Market Comparison Wages

Sources: News reporting, public job postings, BLS data
- Buc-ee's: $18+/hr entry (widely reported)
- Amazon warehouse: ~$17-18/hr (public)
- H-E-B: $15+/hr (reported)
- Walmart: ~$14/hr (public)
- Fast food: $12-13/hr (BLS + reporting)

In [ ]:
market_wages = pd.DataFrame([
    {'employer': 'HHSC Target (old)', 'wage': 10.60, 'category': 'medicaid_assumption', 'source': 'PFD Wage Calculator B7'},
    {'employer': 'HHSC Target (new)', 'wage': 13.00, 'category': 'medicaid_assumption', 'source': 'KERA 2025-03-04'},
    {'employer': 'BLS TX Home Health Aide (mean)', 'wage': hha_mean, 'category': 'bls_actual', 'source': 'BLS OEWS May 2024'},
    {'employer': 'BLS TX Home Health Aide (median)', 'wage': hha_median, 'category': 'bls_actual', 'source': 'BLS OEWS May 2024'},
    {'employer': 'BLS TX Nursing Assistant (mean)', 'wage': 17.79, 'category': 'bls_actual', 'source': 'BLS OEWS May 2024'},
    {'employer': 'TX Minimum Wage', 'wage': 7.25, 'category': 'minimum', 'source': 'Texas law'},
    {'employer': 'Federal Minimum Wage', 'wage': 7.25, 'category': 'minimum', 'source': 'Federal law'},
    {'employer': "Buc-ee's (entry)", 'wage': 18.00, 'category': 'retail', 'source': 'Public job postings / news'},
    {'employer': 'Amazon Warehouse', 'wage': 17.50, 'category': 'retail', 'source': 'Amazon public wage data'},
    {'employer': 'H-E-B (cashier)', 'wage': 15.00, 'category': 'retail', 'source': 'News reporting'},
    {'employer': 'Walmart', 'wage': 14.00, 'category': 'retail', 'source': 'Walmart public wage data'},
    {'employer': "McDonald's (TX avg)", 'wage': 12.50, 'category': 'food_service', 'source': 'BLS / news reporting'},
    {'employer': 'Whataburger', 'wage': 13.00, 'category': 'food_service', 'source': 'News reporting'},
])

market_wages = market_wages.sort_values('wage', ascending=False)
market_wages.to_csv(PROCESSED / 'market_wages.csv', index=False)

print("=== Wage Comparison ===")
for _, row in market_wages.iterrows():
    bar = '█' * int(row['wage'] * 2)
    print(f"  {row['employer']:<40} ${row['wage']:>6.2f}  {bar}")

---
## Summary: What We Have

| Dataset | File | Records | Key Finding |
|---|---|---|---|
| PFD Wage Calculator | `pfd_wage_calculator_all_services.csv` | ~350 service lines | $10.60 default target wage |
| PFD LTSS Rate Tables | `2026-2027-ltss-rte-tables.xlsx` | Multiple sheets | Current Medicaid rates by service |
| BLS OEWS Texas | `bls_oews_texas_2024.csv` | 11 measures | TX HHA mean = $12.19/hr |
| Market Wages | `market_wages.csv` | 13 comparisons | Buc-ee's pays $5+ more than care work |

In [ ]:
print("\nProcessed files:")
for f in sorted(PROCESSED.glob('*.csv')):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")